# Training MNB model

In [8]:
import pandas as pd
import numpy as np
%pip install scikit-learn
from sklearn.model_selection import train_test_split

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\Wiltj\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
df = pd.read_csv("data/merged_data.csv", index_col="date", parse_dates=True)
df.sort_index(inplace=True)

We define our features (X) and target (y)

Our features (X) is comprised of our cleaned text data (X1) and our financial data (X2).

In [10]:
X = df[["summary", "pct_change", "volume"]]
y = df["target"]

Now we split our data for training and testing

DO NOT SHUFFLE because we don't want to mix time series data

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [12]:
print(f"Training Range: {X_train.index.min()} to {X_train.index.max()}")
print(f"Testing Range:  {X_test.index.min()} to {X_test.index.max()}")

Training Range: 2017-12-19 00:00:00 to 2020-01-10 00:00:00
Testing Range:  2020-01-13 00:00:00 to 2020-07-17 00:00:00


Now we have two different types of data that we need to handle differently

1. Text data - we will use TF-IDF vectorization to convert text into numerical format
2. Numerical data - we will use MinMaxScaler to normalize the data

The reason we need two different scalers is because Naive Bayes won't work with negative values

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

text_features = "summary"
numerical_features = ["pct_change", "volume"]

preprocessor = ColumnTransformer(
    transformers=[
        ("tfidf", TfidfVectorizer(max_features=5000), text_features),
        ("scaler", MinMaxScaler(), numerical_features)
    ]
)

In [14]:
# Fit on train
X_train_transformed = preprocessor.fit_transform(X_train)

# Transform test
X_test_transformed = preprocessor.transform(X_test)

In [15]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Model
model = MultinomialNB(alpha=0.1)  # smoothing matters
model.fit(X_train_transformed, y_train)

# Predict
y_pred = model.predict(X_test_transformed)


In [16]:
# Results
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.5461538461538461
              precision    recall  f1-score   support

         0.0       0.33      0.02      0.03        58
         1.0       0.55      0.97      0.70        72

    accuracy                           0.55       130
   macro avg       0.44      0.49      0.37       130
weighted avg       0.45      0.55      0.40       130



The model predicts up most of the tie and guesses it right 54% of the time it is not good at guessing the down trend in the market